# Эксперименты и итоговое сравнение

У Grounding DINO меняем постобработку рамок. У Qwen сравниваем исходный и уточнённый промпты. Ноутбук по умолчанию читает опубликованные предсказания и пересчитывает метрики на CPU. В режиме `RERUN_ALL=True` все модели запускаются отдельными ячейками в одном окружении. Новые предсказания и таблицы пишутся в `runs/production_validation/<устройство>/`. Установка зависимостей для CPU и CUDA описана в README.

In [1]:
import sys
from pathlib import Path
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = (cwd, *cwd.parents, Path("/kaggle/working/data-labeling-shad"))
ROOT = next((p for p in candidates if (p / "model_comparison/workflow.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Открой ноутбук из клона репозитория или склонируй его в /kaggle/working/data-labeling-shad")
sys.path.insert(0, str(ROOT))
print("Корень репозитория найден")

Корень репозитория найден


In [2]:
from model_comparison.workflow import (
    RESULTS, baseline, compare_experiments, deduplicate, group_table, leaderboard, load_validation, measure,
    qwen_prompt_variant, check_production_runtime,
)
from model_comparison.models.metrics import evaluate
from model_comparison.models.pipeline import read_jsonl, write_jsonl

photos, golden = load_validation()
print(f"Общий датасет: {len(photos)} фото")

Общий датасет: 400 фото


In [3]:
RERUN_ALL = False
DEVICE = check_production_runtime("auto") if RERUN_ALL else None
RUN_DIR = ROOT / "runs/production_validation" / (DEVICE or "replay")
GROUP_COLUMNS = ["n_photos", "f1_50", "f1_75", "precision_50", "recall_50",
                "fp_50", "fn_50", "negative_photos_with_fp", "fp_on_negative_photos",
                "technical_errors"]
TEAM_COLUMNS = GROUP_COLUMNS + ["inference_seconds", "ms_per_photo"]
print("Источник:", f"новый инференс на {DEVICE}" if RERUN_ALL else "сохранённые запуски")

Источник: сохранённые запуски


## Grounding DINO — удаление дублей

**Гипотеза.** Почти одинаковые рамки часто соответствуют одной бабочке. Если оставлять первую и убирать последующие рамки с IoU ≥ 0.85 с уже оставленной, число FP уменьшится, precision вырастет, а recall существенно не изменится.

**Проверка.** Используем те же предсказания, все 400 фото и ту же разметку. Меняем только постобработку рамок. Порог уверенности, порядок кандидатов и модель сохраняются.

In [4]:
dino_pred, dino_seconds = baseline(
    "grounding_dino", photos, rerun=RERUN_ALL, device=DEVICE, output_dir=RUN_DIR / "grounding_dino",
)
dino_result = measure("grounding_dino", photos, golden, dino_pred, dino_seconds)
print(f"Бейзлайн Grounding DINO: {len(dino_pred)} предсказаний")

Бейзлайн Grounding DINO: 400 предсказаний


In [5]:
dino_dedup_pred = deduplicate(dino_pred, threshold=0.85)
if RERUN_ALL:
    write_jsonl(RUN_DIR / "grounding_dino/grounding_dino__dedup085.jsonl", dino_dedup_pred.values())
else:
    saved_dino = read_jsonl(RESULTS / "experiments/dino/grounding_dino__dedup085.jsonl")
    assert dino_dedup_pred == saved_dino
dino_dedup_result = measure("grounding_dino__dedup085", photos, golden, dino_dedup_pred, dino_seconds)
display(leaderboard(dino_result, dino_dedup_result)[TEAM_COLUMNS].round(4))

,n_photos,f1_50,f1_75,precision_50,recall_50,fp_50,fn_50,negative_photos_with_fp,fp_on_negative_photos,technical_errors,inference_seconds,ms_per_photo
run,,,,,,,,,,,,
grounding_dino__dedup085,400,0.5854,0.5366,0.4259,0.9358,275,14,182,239,0,199.2971,498.2427
grounding_dino,400,0.3788,0.3510,0.2375,0.9358,655,14,182,341,0,199.2971,498.2427


**Вывод по сохранённому запуску.** Число FP снизилось с 655 до 275, precision вырос с 0.2375 до 0.4259, recall остался 0.9358. F1@0.50 вырос с 0.3788 до 0.5854. Число негативных кадров хотя бы с одним FP осталось 182. Удаление дублей убрало повторные рамки, а ошибки распознавания фона сохранились. Время в таблице относится к общему исходному инференсу.

## Qwen — уточнённый промпт

**Гипотеза.** Правило возвращать пустой список на фото с цветами или другими насекомыми сократит FP на негативных кадрах при близком recall. Слишком строгое условие может увеличить число пропусков на сложных кадрах. Сравниваем F1@0.50, precision, recall и FP на тех же 400 фото.

**Проверка.** Сохранённые запуски используют одну ревизию весов и один датасет. Версии PyTorch и Transformers между ними различаются. В новом прогоне безусловно используется окружение продакшена и один конфиг Qwen. Для бейзлайна в нём меняется только промпт.

In [6]:
qwen_ref_pred, qwen_ref_seconds = qwen_prompt_variant(
    photos, strict=False, rerun=RERUN_ALL, device=DEVICE, output_dir=RUN_DIR / "qwen",
)
qwen_ref = measure("qwen__reference", photos, golden, qwen_ref_pred, qwen_ref_seconds)
print(f"Исходный промпт: {len(qwen_ref_pred)} предсказаний")

Исходный промпт: 400 предсказаний


In [7]:
qwen_strict_pred, qwen_strict_seconds = qwen_prompt_variant(
    photos, strict=True, rerun=RERUN_ALL, device=DEVICE, output_dir=RUN_DIR / "qwen",
)
qwen_strict = measure("qwen__strict_negative_prompt", photos, golden, qwen_strict_pred, qwen_strict_seconds)
print(f"Уточнённый промпт: {len(qwen_strict_pred)} предсказаний")

Уточнённый промпт: 400 предсказаний


In [8]:
display(leaderboard(qwen_ref, qwen_strict)[TEAM_COLUMNS].round(4))
display(group_table(qwen_ref, qwen_strict)[GROUP_COLUMNS].round(4))

,n_photos,f1_50,f1_75,precision_50,recall_50,fp_50,fn_50,negative_photos_with_fp,fp_on_negative_photos,technical_errors,inference_seconds,ms_per_photo
run,,,,,,,,,,,,
qwen__strict_negative_prompt,400,0.8742,0.7889,0.8167,0.9404,46,13,43,45,0,NaN,NaN
qwen__reference,400,0.6508,0.5841,0.4976,0.9404,207,13,202,206,0,953.0246,2382.5615


n_photos  ...  technical_errors
run                          group                         ...                  
qwen__reference              butterfly                191  ...                 0
                             insect                   101  ...                 0
                             multiple_butterfly         6  ...                 0
                             no_butterfly             102  ...                 0
qwen__strict_negative_prompt butterfly                191  ...                 0
                             insect                   101  ...                 0
                             multiple_butterfly         6  ...                 0
                             no_butterfly             102  ...                 0

[8 rows x 10 columns]

**Вывод по сохранённым запускам.** Число негативных кадров с FP снизилось с 202 до 43 из 203, precision вырос с 0.4976 до 0.8167, общий recall остался 0.9404. В группе `insect` остались FP на 40 из 101 фото, в `no_butterfly` — на 3 из 102. F1@0.50 вырос с 0.6508 до 0.8742. Наблюдаемый прирост относится к двум сохранённым конфигурациям целиком, поскольку версии библиотек различались.

## Итоговый лидерборд и результаты по группам

In [9]:
owl_pred, owl_seconds = baseline("owlv2", photos, rerun=RERUN_ALL, device=DEVICE, output_dir=RUN_DIR / "owlv2")
owl_result = measure("owlv2", photos, golden, owl_pred, owl_seconds)
print(f"Бейзлайн OWLv2: {len(owl_pred)} предсказаний")

Бейзлайн OWLv2: 400 предсказаний


In [10]:
qwen_base = measure("qwen", photos, golden, qwen_ref_pred, qwen_ref_seconds)
print(f"Исходный промпт учтён как бейзлайн Qwen: {len(qwen_ref_pred)} предсказаний")

Исходный промпт учтён как бейзлайн Qwen: 400 предсказаний


In [11]:
runs = [dino_result, owl_result, qwen_base, dino_dedup_result, qwen_strict]

team_table = leaderboard(*runs)
display(team_table[TEAM_COLUMNS].round(4))

,n_photos,f1_50,f1_75,precision_50,recall_50,fp_50,fn_50,negative_photos_with_fp,fp_on_negative_photos,technical_errors,inference_seconds,ms_per_photo
run,,,,,,,,,,,,
qwen__strict_negative_prompt,400,0.8742,0.7889,0.8167,0.9404,46,13,43,45,0,NaN,NaN
qwen,400,0.6508,0.5841,0.4976,0.9404,207,13,202,206,0,953.0246,2382.5615
grounding_dino__dedup085,400,0.5854,0.5366,0.4259,0.9358,275,14,182,239,0,199.2971,498.2427
owlv2,400,0.4760,0.3486,0.3146,0.9771,464,5,103,191,0,337.0020,842.5050
grounding_dino,400,0.3788,0.3510,0.2375,0.9358,655,14,182,341,0,199.2971,498.2427


In [12]:
by_group = group_table(*runs)
display(by_group[GROUP_COLUMNS].round(4))

n_photos  ...  technical_errors
run                          group                         ...                  
grounding_dino               butterfly                191  ...                 0
                             insect                   101  ...                 0
                             multiple_butterfly         6  ...                 0
                             no_butterfly             102  ...                 0
owlv2                        butterfly                191  ...                 0
                             insect                   101  ...                 0
                             multiple_butterfly         6  ...                 0
                             no_butterfly             102  ...                 0
qwen                         butterfly                191  ...                 0
                             insect                   101  ...                 0
                             multiple_butterfly         6  ...                 0
                             no_butterfly             102  ...                 0
grounding_dino__dedup085     butterfly                191  ...                 0
                             insect                   101  ...                 0
                             multiple_butterfly         6  ...                 0
                             no_butterfly             102  ...                 0
qwen__strict_negative_prompt butterfly                191  ...                 0
                             insect                   101  ...                 0
                             multiple_butterfly         6  ...                 0
                             no_butterfly             102  ...                 0

[20 rows x 10 columns]

In [13]:
comparison = compare_experiments(dino_result, dino_dedup_result, qwen_ref, qwen_strict)
display(comparison[["f1_50_before", "f1_50_after", "f1_50_delta",
                    "fp_50_before", "fp_50_after"]].round(4))

,f1_50_before,f1_50_after,f1_50_delta,fp_50_before,fp_50_after
experiment,,,,,
Grounding DINO: удаление дублей,0.3788,0.5854,0.2065,655,275
Qwen: уточнённый промпт,0.6508,0.8742,0.2234,207,46


In [14]:
table_dir = RUN_DIR / "tables" if RERUN_ALL else ROOT / "runs/tables"
table_dir.mkdir(parents=True, exist_ok=True)
leaderboard(dino_result, owl_result, qwen_base).to_csv(table_dir / "leaderboard_baseline.csv")
team_table.to_csv(table_dir / "leaderboard.csv")
by_group.to_csv(table_dir / "leaderboard_by_group.csv")
comparison.to_csv(table_dir / "experiments_comparison.csv")
print("Таблицы сохранены в", table_dir.relative_to(ROOT))

Таблицы сохранены в runs/tables


## Примеры ошибок

In [15]:
import pandas as pd

example_ids = [669699330, 760875513, 761031648]
_, before = evaluate(example_ids, golden, qwen_ref_pred)
_, after = evaluate(example_ids, golden, qwen_strict_pred)
cases = pd.DataFrame([
    {"photo_id": pid, "группа": photos[pid]["group"], "рамок в эталоне": len(golden[pid]["boxes"]),
     "FP до": old["fp_50"], "FP после": new["fp_50"], "FN после": new["fn_50"]}
    for pid, old, new in zip(example_ids, before, after)
]).set_index("photo_id")
display(cases)
for pid in example_ids:
    print(pid, photos[pid]["image"])

669699330 https://inaturalist-open-data.s3.amazonaws.com/photos/669699330/original.jpg
760875513 https://inaturalist-open-data.s3.amazonaws.com/photos/760875513/original.jpg
761031648 https://inaturalist-open-data.s3.amazonaws.com/photos/761031648/original.jpg


,группа,рамок в эталоне,FP до,FP после,FN после
photo_id,,,,,
669699330,no_butterfly,0,1,0,0
760875513,insect,0,1,0,0
761031648,multiple_butterfly,9,1,1,7


В сохранённом запуске на двух негативных кадрах из таблицы строгий Qwen убрал лишнюю рамку. На фото 761031648 он пропустил 7 из 9 бабочек при IoU ≥ 0.50. В выборке всего 6 фото с несколькими бабочками, поэтому результат этой группы чувствителен к отдельным кадрам.

**Выбор.** Для Контеста использован строгий Qwen с наибольшим F1@0.50 на командной валидации (0.8742, F1@0.75 — 0.7889). На этапе оценки 195 пустых кадров прошли предварительную фильтрацию без рамочного ревью. Часть остальных рамок взята из предразметки после проверки. Эти особенности эталона учитываем при интерпретации ложных срабатываний.

In [1]:
tables = table_dir
output = tables.parent / "RESULTS.md"

model_names = {
    "grounding_dino": "Grounding DINO Tiny",
    "grounding_dino__dedup085": "Grounding DINO + удаление дублей",
    "owlv2": "OWLv2 Base",
    "qwen": "Qwen3-VL-2B",
    "qwen__strict_negative_prompt": "Qwen3-VL-2B + уточнённый промпт",
}

labels = {
    "run": "Конфигурация",
    "n_photos": "Фото",
    "f1_50": "F1@0.50",
    "f1_75": "F1@0.75",
    "precision_50": "Precision@0.50",
    "recall_50": "Recall@0.50",
    "fp_50": "FP",
    "fn_50": "FN",
    "technical_errors": "Тех. ошибки",
    "n_negative_photos": "Негативные фото",
    "negative_photos_with_fp": "Негативные фото с FP",
    "fp_on_negative_photos": "FP на негативных фото",
    "inference_seconds": "Инференс, с",
    "ms_per_photo": "мс/фото",
}

metrics = [
    "n_photos", "f1_50", "f1_75", "precision_50", "recall_50",
    "fp_50", "fn_50", "technical_errors",
    "n_negative_photos", "negative_photos_with_fp", "fp_on_negative_photos",
]
timing = ["inference_seconds", "ms_per_photo"]


def read_table(filename):
    df = pd.read_csv(tables / filename)
    if "run" in df:
        df["run"] = df["run"].replace(model_names)
    return df


def markdown_table(df):
    def format_value(value):
        if pd.isna(value):
            return "—"
        if isinstance(value, float):
            return f"{value:.4f}".rstrip("0").rstrip(".")
        return str(value).replace("|", r"\|")

    def format_row(values):
        return "| " + " | ".join(map(format_value, values)) + " |"

    return "\n".join([
        format_row(df.columns),
        format_row(["---"] * len(df.columns)),
        *(format_row(row) for row in df.itertuples(index=False, name=None)),
    ])


sections = [
    "# Результаты детекции бабочек",
    "FP и FN указаны при IoU ≥ 0.50. Прочерк означает, что значение отсутствует "
    "в исходной таблице. Время DINO после удаления дублей относится к исходному "
    "инференсу. " + (
        "Прогоны двух промптов Qwen выполнены в одном окружении."
        if RERUN_ALL else
        "Сохранённые прогоны двух промптов Qwen выполнены в разных окружениях."
    ),
]

for title, filename in [
    ("Бейзлайны", "leaderboard_baseline.csv"),
    ("Итоговый лидерборд", "leaderboard.csv"),
]:
    df = read_table(filename)[["run", *metrics, *timing]].rename(columns=labels)
    sections.append(f"## {title}\n\n{markdown_table(df)}")

sections.append("## Эксперименты")
experiment_metrics = [
    "f1_50", "f1_75", "precision_50", "recall_50", "fp_50", "fn_50",
    "negative_photos_with_fp", "technical_errors",
]
for experiment in read_table("experiments_comparison.csv").to_dict("records"):
    changes = pd.DataFrame([
        {
            "Метрика": labels[metric],
            "До": experiment[f"{metric}_before"],
            "После": experiment[f"{metric}_after"],
            "Δ": experiment[f"{metric}_delta"],
        }
        for metric in experiment_metrics
    ])
    sections.append(f"### {experiment['experiment']}\n\n{markdown_table(changes)}")

sections.append("## Результаты по группам")
for group, df in read_table("leaderboard_by_group.csv").groupby("group", sort=False):
    df = df[["run", *metrics]].rename(columns=labels)
    sections.append(f"### {group}\n\n{markdown_table(df)}")

output.write_text("\n\n".join(sections) + "\n", encoding="utf-8")
print(f"Готово: {output.relative_to(ROOT)}")

Готово: model_comparison/results/RESULTS.md
